# 45-1 Análisis de Sentimiento con PySpark

Este notebook demuestra cómo realizar análisis de sentimiento usando PySpark ML y TextBlob, sin dependencias de Spark NLP (John Snow Labs).

## Objetivo
Clasificar textos en inglés como positivos, negativos o neutrales usando dos enfoques:
1. **TextBlob** (via UDF): análisis basado en polaridad léxica (rápido, sin entrenamiento).
2. **PySpark ML** (TF-IDF + Logistic Regression): clasificación supervisada entrenada sobre datos etiquetados.

## Flujo del notebook
1. Instalación de dependencias.
2. Importación de módulos PySpark y TextBlob.
3. Creación de la sesión Spark.
4. Creación de datos de ejemplo con sentimiento esperado.
5. Análisis de sentimiento con TextBlob (basado en polaridad).
6. Clasificación con PySpark ML: TF-IDF + Logistic Regression.
7. Evaluación de exactitud del modelo.
8. Prueba con nuevos textos.

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instalan librerías para visualización, manejo de datos y análisis de sentimiento con TextBlob. No se requiere Spark NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn wordcloud textblob

## 2. Importación de módulos
Se importan PySpark (sesión, tipos, funciones, Pipeline ML) y TextBlob para análisis de polaridad.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, udf, expr, lower, regexp_replace, trim, split, explode
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StringIndexer
from pyspark.ml.classification import LogisticRegression
from textblob import TextBlob
import os

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("analisis_sentimiento") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")

## 4. Datos de ejemplo
Se crean oraciones en inglés con su sentimiento esperado (positive, negative, neutral) para probar ambos enfoques. TextBlob funciona nativamente con inglés, por lo que no se necesita traducción.


In [ ]:
data_sentimiento = [
    ("I think Alex Johnson is doing a fantastic job leading the country.", "positive"),
    ("Alex Johnson's policies are ruining our economy.", "negative"),
    ("I'm not sure about Alex Johnson's latest speech, it was confusing.", "neutral"),
    ("The new reforms introduced by Alex Johnson are very promising.", "positive"),
    ("Alex Johnson seems to care about the people's issues, which is refreshing.", "positive"),
    ("I am disappointed with Alex Johnson's performance.", "negative"),
    ("Alex Johnson's leadership style is quite effective.", "positive"),
    ("The way Alex Johnson handled the recent crisis was commendable.", "positive"),
    ("I don't trust Alex Johnson's intentions at all.", "negative"),
    ("Alex Johnson has brought positive changes to the healthcare system.", "positive")
]
df_sentimiento = spark.createDataFrame(data_sentimiento, ["texto", "sentimiento_esperado"])
df_sentimiento.show(truncate=False)

## 5. Análisis de sentimiento con TextBlob (basado en polaridad)
Se usa `TextBlob` via UDF para calcular la polaridad de cada texto directamente en inglés. La polaridad va de -1 (negativo) a +1 (positivo) y se clasifica según umbrales.

> **Nota:** Este enfoque es léxico (basado en diccionario) y no requiere entrenamiento. Es una buena primera aproximación.


In [ ]:
# UDF: calcular polaridad con TextBlob 
@udf(StringType())
def sentiment_textblob(text):
    if text is None:
        return "neutral"
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    else:
        return "neutral"

@udf(DoubleType())
def polarity_textblob(text):
    if text is None:
        return 0.0
    return float(TextBlob(text).sentiment.polarity)

# Aplicar TextBlob al DataFrame
df_tb = df_sentimiento.withColumn("sentimiento_textblob", sentiment_textblob(col("texto"))) \
                      .withColumn("polaridad", polarity_textblob(col("texto")))

df_tb.select("texto", "sentimiento_esperado", "sentimiento_textblob", "polaridad").show(truncate=False)

## 6. Clasificación supervisada con PySpark ML: TF-IDF + Logistic Regression
Ahora se usa un enfoque de Machine Learning clásico:
1. **Tokenizer**: divide el texto en palabras.
2. **HashingTF**: convierte tokens a features de frecuencia de términos.
3. **IDF**: aplica TF-IDF para ponderar por importancia.
4. **StringIndexer**: convierte las etiquetas de sentimiento a índices numéricos.
5. **LogisticRegression**: entrena un clasificador sobre las features TF-IDF.


In [ ]:
# Preparar etiquetas numéricas
indexer = StringIndexer(inputCol="sentimiento_esperado", outputCol="label").fit(df_sentimiento)
df_ml = indexer.transform(df_sentimiento)

# Pipeline: Tokenizer → HashingTF → IDF → LogisticRegression
tokenizer_ml = Tokenizer(inputCol="texto", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="tf", numFeatures=256)
idf = IDF(inputCol="tf", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

pipeline_lr = Pipeline(stages=[tokenizer_ml, hashingTF, idf, lr])
model_lr = pipeline_lr.fit(df_ml)
df_pred = model_lr.transform(df_ml)

# Mostrar predicciones junto con etiqueta esperada
df_pred.select("texto", "sentimiento_esperado", "label", "prediction").show(truncate=False)

## 7. Evaluación de exactitud del modelo
Se calcula la proporción de predicciones correctas sobre el total. Se comparan ambos enfoques (TextBlob y Logistic Regression).

> **Nota:** Con tan pocos datos de entrenamiento, la exactitud del modelo supervisado puede ser perfecta por sobreajuste. En un caso real se usarían conjuntos de train/test separados.


In [ ]:
# Exactitud del modelo Logistic Regression
correctas_lr = df_pred.filter(expr("prediction == label")).count()
total = df_pred.count()
print(f"Exactitud Logistic Regression: {correctas_lr/total:.2f}")

# Exactitud de TextBlob
correctas_tb = df_tb.filter(col("sentimiento_esperado") == col("sentimiento_textblob")).count()
print(f"Exactitud TextBlob: {correctas_tb/total:.2f}")

## 8. Prueba con nuevos textos
Se aplica el modelo entrenado a textos nuevos para evaluar cómo generaliza la clasificación.


In [ ]:
# Nuevos textos para probar
nuevos_textos = [
    ("The service was quick and friendly, I highly recommend it.",),
    ("I didn't like it at all, a complete waste of time.",),
    ("The price is reasonable, nothing special though.",),
]
df_nuevos = spark.createDataFrame(nuevos_textos, ["texto"])

# Predicción con TextBlob
df_nuevos_tb = df_nuevos.withColumn("sentimiento_textblob", sentiment_textblob(col("texto"))) \
                        .withColumn("polaridad", polarity_textblob(col("texto")))
df_nuevos_tb.show(truncate=False)